# Cardio Descriptive Analytics

本Notebook用于复现开发集的描述性分析。只使用train.csv，最终测试集不参与探索。

方法：描述性统计、分组聚合、交叉表、箱线图与Spearman相关。小于100条的组隐藏比例。类别关联不代表因果关系。

先安装项目根目录requirements_descriptive.txt中的依赖。Notebook尚未执行，已有报告和图表由正式脚本实际生成。


In [ ]:
from pathlib import Path
import csv, json, subprocess, sys
from IPython.display import display, Image, HTML

root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/processed/train.csv").exists()), None)
if root is None:
    raise RuntimeError("请从final project目录或其notebooks子目录打开Notebook")
print("Project:", root)


## 1 数据范围
每行是一条健康记录，cardio为疾病状态标签。


In [ ]:
with (root / "data/processed/train.csv").open(encoding="utf-8-sig", newline="") as f:
    records = list(csv.DictReader(f))
assert all(r["split"] == "train" for r in records)
print("Development records:", len(records))
print("Label-1 proportion:", sum(int(r["cardio"]) for r in records) / len(records))


## 2 运行统计和制图
调用项目正式分析脚本，生成并验证全部CSV、PNG、SVG、HTML和中文报告。


In [ ]:
result = subprocess.run([sys.executable, str(root / "src/descriptive_analysis.py"), "--project-root", str(root)], check=True, capture_output=True, text=True)
print(result.stdout)


## 3 分组比例与连续指标
所有比例以当前组记录数为分母。注意胆固醇、血糖为等级，gender暂保留代码。


In [ ]:
import html
def show_csv(name):
    with (root / "outputs/descriptive/tables" / name).open(encoding="utf-8-sig", newline="") as f:
        values = list(csv.DictReader(f))
    headers = list(values[0])
    text = "<table><tr>" + "".join("<th>"+html.escape(c)+"</th>" for c in headers) + "</tr>"
    for row in values:
        text += "<tr>" + "".join("<td>"+html.escape(str(row[c]))+"</td>" for c in headers) + "</tr>"
    display(HTML(text+"</table>"))

show_csv("age_group.csv")
show_csv("bmi_band.csv")
show_csv("numeric_summary.csv")


## 4 核心图表
图表使用开发集；箱线图保留全部数据点，BMI直方图为对数纵轴。


In [ ]:
manifest = json.loads((root / "reports/descriptive/analysis_manifest.json").read_text(encoding="utf-8"))
for figure in manifest["figures"]:
    print(figure["title"])
    display(Image(filename=str(root / "outputs/figures/descriptive" / (figure["file"] + ".png"))))


## 5 主要发现与限制
以实际计算结果为准，不将吸烟或饮酒的粗分组比例解释成保护效应。年龄、性别、选择和测量偏差都可能影响关联。


In [ ]:
print((root / "reports/descriptive/descriptive_report_zh.txt").read_text(encoding="utf-8"))
print((root / "reports/descriptive/validation.txt").read_text(encoding="utf-8"))


## 与预测分析衔接
用开发集比较基础资料和加入体检指标的模型。标准化等步骤仅在训练折拟合，相同profile_group不要跨交叉验证折。test.csv留作最终评估。

数据来源：https://github.com/caravanuden/cardio 。ChatGPT协助生成分析代码和解释，统计结果由本地脚本计算。
